In [2]:
import numpy as np

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

---

In [4]:
# Forward pass - Python
def forwardPy(inp, W, bias):
    z = 0
    for i in range(len(inp)):
        z += (inp[i] * W[i])
    z += bias
    a = sigmoid(z)
    return z, a

In [5]:
# Forward pass - Numpy
def forwardNp(inp, W, bias):
    z = np.dot(inp, W) + bias
    a = sigmoid(z)
    return z, a

In [180]:
# Test forward pass
num_feats = 2
W = np.random.randn(num_feats)
bias = np.random.randn()
inp = np.random.randn(num_feats)

assert forwardPy(inp, W, bias) == forwardNp(inp, W, bias)

---

In [181]:
# MSE loss - Python - works only with lists/tuples
def MSELossPy(preds, targs):
    loss = 0
    len_preds = len(preds)
    for i in range(len_preds):
        loss += (preds[i] - targs[i]) ** 2
    loss /= len_preds
    return loss

assert MSELossPy([0, 0], [0, 1]) == 0.5
assert MSELossPy([1, 1], [1, 1]) == 0
assert MSELossPy([0, 0], [1, 1]) == 1

In [72]:
# MSE loss - Numpy - works with numpy arrays, and scalars
def MSELossNp(preds, targs):
    return np.mean((preds - targs) ** 2)

assert MSELossNp(np.array([0, 0]), np.array([0, 1])) == 0.5
assert MSELossNp(np.array([1, 1]), np.array([1, 1])) == 0
assert MSELossNp(np.array([0, 0]), np.array([1, 1])) == 1
assert MSELossNp(0, 0) == 0
assert MSELossNp(0, 1) == 1


---

In [6]:
# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # (4, 2)
y = np.array([0, 1, 1, 0]) # (4,)
num_samples = len(X) # 4

print(X[0])
print(X[:, 0])
print(X[0, :])

[0 0]
[0 0 1 1]
[0 0]


---

In [7]:
# Model params - One neuron
num_features = X.shape[1]
W = np.random.randn(num_features)
bias = np.random.randn()

print(W)
print(bias)

[1.27817139 0.86321874]
-1.0807339397802225


In [185]:
z, a = forwardNp(X, W, bias)
loss = MSELossNp(a, y)

print("z:", z)
print("a:", a)
print("y:", y)
print("loss:", loss)

z: [ 0.19308752  0.50492439 -0.46788986 -0.15605299]
a: [0.54812246 0.62361588 0.38511581 0.46106573]
y: [0 1 1 0]
loss: 0.2581918545232616


In [186]:
def backprop(X, y, a):
    dL_da = 2 * (a - y) # (4,)
    da_dz = a * (1 - a) # (4,)
    dz_dW = X # (4, 2)
    dL_dz = dL_da * da_dz # (4,)

    num_samples = len(X)
    dL_dW = np.dot(dz_dW.T, dL_dz) / num_samples # (2, 4) @ (4,) -> (2,)

    dL_dB = np.mean(dL_dz) # average over all samples
    return dL_dW, dL_dB

In [187]:
# Train the neuron
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    z, a = forwardNp(X, W, bias)

    # loss
    loss = MSELossNp(a, y)

    # backward
    gradW, gradB = backprop(X, y, a)

    # update
    W = W - lr * gradW
    bias = bias - lr * gradB

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.2581918545232616
loss at epoch 500: 0.2561814994805921
loss at epoch 1000: 0.2546833266789315
loss at epoch 1500: 0.25353387397607746
loss at epoch 2000: 0.2526563644825723
loss at epoch 2500: 0.2519920388693464
loss at epoch 3000: 0.2514923593112212
loss at epoch 3500: 0.2511181236267235
loss at epoch 4000: 0.25083852789274597
loss at epoch 4500: 0.2506298595197188


In [188]:
# Test the neuron
_, a = forwardNp([0,0], W, bias)
print(a)
_, a = forwardNp([0,1], W, bias)
print(a)
_, a = forwardNp([1,0], W, bias)
print(a)
_, a = forwardNp([1,1], W, bias)
print(a)

0.5178559695464336
0.5284637194098327
0.4759402249900127
0.4865558313799513


---

In [75]:
# Model params - Two neurons
num_features = X.shape[1]
W1 = np.random.randn(num_features)
b1 = np.random.randn()
W2 = np.random.randn()
b2 = np.random.randn()

print(f"W1: {W1}\nW2: {W2}")
print(f"b1: {b1}\nb2: {b2}")

W1: [ 0.88326264 -0.39321855]
W2: 1.1177487169682334
b1: -0.9258039405994264
b2: 1.8253013684946269


In [76]:
# Forward pass
def forward_two_neurons(W1, b1, W2, b2, X):
    _, a1 = forwardNp(X, W1, b1)
    _, a2 = forwardNp(a1, W2, b2)
    return a1, a2

In [77]:
# Test forward pass
a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)
print(f"a1: {a1}\na2: {a2}")

a1: [0.28377678 0.21098097 0.48936628 0.39275177]
a2: [0.89496514 0.88706693 0.91468972 0.90587683]


In [78]:
# Backward pass
def backward_two_neurons(x, y, a1, a2, w2):
    num_samples = W1.shape[0]   # () -> 4

    dl_da2 = 2 * (a2 - y)       # (4,)
    da2_dz2 = a2 * (1 - a2)     # (4, )
    dz2_dw2 = a1                # (4,)
    dz2_db2 = 1                 # ()

    dz2_da1 = w2                # ()
    da1_dz1 = a1 * (1 - a1)     # (4,)
    dz1_dw1 = x                 # (4, 2)
    dz1_db1 = 1                 # ()

    dl_dz2 = dl_da2 * da2_dz2   # (4,)

    dl_dw2 = np.dot(dl_dz2, dz2_dw2) / num_samples  # () which matches W2's shape
    dl_db2 = np.mean(dl_dz2 * dz2_db2)              # () which matches b2's shape

    dl_dz1 = dl_dz2 * dz2_da1 * da1_dz1                 # (4,) -> the gradient handled back to neuron 1
    dl_dw1 = np.dot(dz1_dw1.T, dl_dz1) / num_samples    # (2,4) @ (4,) -> (2,) which matches W1's shape
    dl_db1 = np.mean(dl_dz1 * dz1_db1)                  # () which matches b1's shape

    return dl_dw2, dl_db2, dl_dw1, dl_db1

In [79]:
# Test backward pass
backward_two_neurons(X, y, a1, a2, W2)

(np.float64(0.04856484356844527),
 np.float64(0.07169857887309977),
 array([0.01873097, 0.01848523]),
 np.float64(0.01786913162785194))

In [80]:
# Train 2 neurons
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)

    # loss
    loss = MSELossNp(a2, y)

    # backward
    dl_dw2, dl_db2, dl_dw1, dl_db1 = backward_two_neurons(X, y, a1, a2, W2)

    # update weights
    W1 = W1 - lr * dl_dw1
    b1 = b1 - lr * dl_db1

    W2 = W2 - lr * dl_dw2
    b2 = b2 - lr * dl_db2

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.41040179224840767
loss at epoch 500: 0.3667642057314908
loss at epoch 1000: 0.31219303933225817
loss at epoch 1500: 0.2721119833516684
loss at epoch 2000: 0.255882301857721
loss at epoch 2500: 0.25121147672099564
loss at epoch 3000: 0.24998811019443173
loss at epoch 3500: 0.24964378260208073
loss at epoch 4000: 0.24951278447667785
loss at epoch 4500: 0.24943076903291794


In [81]:
# Test the 2 neurons
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,1])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,1])
print(a2)

0.5025665017155148
0.4961210094309298
0.5117755946778043
0.5025922874567922


---